### Install Required Libraries
We will install `fastcoref` which is a modern, fast, and highly accurate library for coreference resolution built on top of Hugging Face Transformers and spaCy.

In [5]:
!pip install fastcoref spacy "transformers<4.37.0"
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 100.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.2 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency confl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 102.5 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### Run Coreference Resolution
Here we load the model and resolve coreferences in a sample text.

In [4]:

import logging
import warnings

# Suppress unnecessary warnings
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# Import FastCoref
from fastcoref import FCoref

# Load pretrained Coreference Resolution model
model = FCoref()

# Input text
text = "My sister has a dog. She loves him. He is a golden retriever."

# Perform Coreference Resolution
predictions = model.predict(texts=[text])

# Extract coreference clusters
clusters = predictions[0].get_clusters()

# Resolve pronouns using a fallback helper function
def resolve_references(text, clusters):
    if not clusters:
        return text
    # Sort spans in reverse order to replace without shifting character indices
    spans_to_replace = []
    for cluster in clusters:
        main_mention = cluster[0]
        for mention in cluster[1:]:
            # Find span coordinates
            start_idx = text.find(mention)
            if start_idx != -1:
                end_idx = start_idx + len(mention)
                spans_to_replace.append((start_idx, end_idx, main_mention))

    spans_to_replace = sorted(spans_to_replace, key=lambda x: x[0], reverse=True)
    resolved = list(text)
    for start, end, replacement in spans_to_replace:
        resolved[start:end] = list(replacement)
    return "".join(resolved)

# Get resolved text
try:
    resolved_text = predictions[0].get_resolved_text()
except AttributeError:
    resolved_text = resolve_references(text, clusters)

# Display results
print("========== COREFERENCE RESOLUTION ==========")

print("\nOriginal Text:")
print(text)

print("\nIdentified Clusters:")
for cluster in clusters:
    print(cluster)

print("\nResolved Text:")
print(resolved_text)


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Inference:   0%|          | 0/1 [00:00<?, ?it/s]

========== COREFERENCE RESOLUTION ==========

Original Text:
My sister has a dog. She loves him. He is a golden retriever.

Identified Clusters:
['a dog', 'him', 'He']

Resolved Text:
My sister has a dog. She loves a dog. a dog is a golden retriever.
